# 08 — Local image ingestion and multimodal model boundary

This notebook uses two original **stylized illustrations**, not photographs of food. It reads the same manifest and invokes the same Python functions as `savoroute media ingest`. The default workflow is offline and needs only Pillow. CLIP inference is opt-in because downloading weights is separate from the testable ingestion pipeline.

## 1. Inspect the catalog and media manifest

Each image has a stable `media_id`, a parent `item_id`, a relative source path, and declared rights metadata. Rights declarations are recorded, not independently verified.

In [ ]:
import json
from pathlib import Path
from savoroute.data.catalog import load_catalog
from savoroute.media.ingestion import parse_image_manifest

catalog = load_catalog()
manifest_path = Path("data/sample/image_manifest.json")
manifest = parse_image_manifest(json.loads(manifest_path.read_text()), catalog)
print("Catalog items:", len(catalog.items), "media:", len(catalog.media))
for image in manifest.sources:
    print(image.media_id, "→", image.item_id, image.path, image.rights_basis)
assert len(manifest.sources) == 2

## 2. Validate and normalize source images

The importer decodes each local image, applies EXIF orientation, converts it to RGB PNG, and removes source image metadata. It rejects unsupported formats, excessive sizes, duplicate pixel content, and paths outside the manifest directory. Input hashes and normalized PNG hashes are stored separately.

### Source illustrations

The two input assets are original, deliberately simplified drawings. They exercise image decoding and catalog references; they do not represent real restaurant or recipe photos.

In [ ]:
from IPython.display import display
from PIL import Image

for source in manifest.sources:
    with Image.open(manifest_path.parent / source.path) as illustration:
        display(illustration.copy())

In [ ]:
from savoroute.media.ingestion import prepare_image_batch

batch = prepare_image_batch(manifest_path, catalog)
for image in batch.images:
    print(image.record.media_id, image.input_format, f"{image.width}x{image.height}",
          image.input_sha256[:12], image.normalized_sha256[:12])
assert len(batch.catalog.media) == 2
assert batch.catalog.media[0].uri == "assets/image_p001.png"

## 3. Publish a self-contained export

The output directory contains `catalog.json`, `provenance.json`, and `assets/`. Catalog media URIs are relative to that output directory. Creating a new output directory avoids silently overwriting prior results.

In [ ]:
from tempfile import TemporaryDirectory
from savoroute.media.ingestion import write_image_batch

with TemporaryDirectory(prefix="savoroute-notebook-") as tmp:
    export = write_image_batch(batch, Path(tmp) / "export")
    exported = load_catalog(export / "catalog.json")
    report = json.loads((export / "provenance.json").read_text())
    print("Media in exported catalog:", len(exported.media))
    for media in exported.media:
        path = export / media.uri
        assert path.is_file()
        print(media.media_id, path.name, path.stat().st_size, "bytes")
    assert len(report["assets"]) == 2

## 4. Separate the embedding model from ingestion

`ClipTextImageEmbedder` implements `JointTextImageEmbedder`: both text and images are embedded by one model into a shared coordinate space. It is not required for ingestion. Inference is off by default; set `SAVOROUTE_RUN_CLIP=1` only after installing the `multimodal` extra and caching the model, or when you intentionally choose to allow a download.

In [ ]:
import os

if os.environ.get("SAVOROUTE_RUN_CLIP") == "1":
    from savoroute.providers.clip import ClipTextImageEmbedder
    from savoroute.media.ingestion import write_image_batch
    from tempfile import TemporaryDirectory
    import math

    encoder = ClipTextImageEmbedder(local_files_only=True)
    with TemporaryDirectory(prefix="savoroute-clip-") as tmp:
        root = write_image_batch(batch, Path(tmp) / "export")
        texts = ["a bowl of noodles", "mushroom pasta"]
        text_vectors = encoder.embed_texts(texts)
        image_vectors = encoder.embed_images([root / media.uri for media in batch.catalog.media])
        # The adapter returns normalized vectors; inner products are cosine similarities.
        for i, text in enumerate(texts):
            similarities = [sum(a * b for a, b in zip(text_vectors[i], image))
                            for image in image_vectors]
            print(text, similarities)
else:
    print("CLIP step skipped; set SAVOROUTE_RUN_CLIP=1 to use cached model weights.")

## 5. Scope and limitations

This milestone establishes validated, traceable *local* image ingestion and an optional joint-embedding adapter. The illustrations and their captions are not a semantic-retrieval benchmark. Vector persistence, approximate nearest neighbors, multimodal fusion, and real-image rights verification are not included in this notebook.